
### 1. Module 1: Entry Discovery & Metadata Promber 1
#### 1.1. Query Logic & Assigned ID Range Traversal
##### 1.1.1. Purpose
Entry discovery use to identify papers within the assigned ranges, retrieve their metadata, and determine the version history of each paper.
##### 1.1.2. Preparing and Traversing Assigned ID Ranges
The prepare_group.py program reads the configuration file `team_ranges.json`, which stores the assigned ID ranges for each team member.

`team_ranges.json`:

```json
{
  "members": [
    {
      "member": "Minh_Hang",
      "ranges": [{"start": "2609.12834", "end": "2609.17833"}]
    },

    {"member": "Trong_Nhan", 
      "ranges": [{"start": "2608.9007", "end": "2608.14006"}]
    },

    {"member": "Vinh_Nguyen", 
      "ranges": [{"start": "2601.18156", "end": "2601.23155"}]
    },

    {"member": "Yen_Vy", 
      "ranges": [{"start": "2511.11709", "end": "2511.16708"}]
    }
  ]
}
```

The `generate_ids()` function in discovery.py generates individual arXiv IDs from the specified ranges. `make_worklist()` merges the generated lists, removes duplicate IDs, and preserves assignment information.

The results are saved into three files:

- `group_ids.txt`: Contains the unique arXiv IDs to be collected.

- `id_ownership.csv`: Maps each arXiv ID to its assigned team member(s).

- `group_summary.json`: Stores summary statistics, including the total number of IDs and assignment information.


Removing duplicate IDs prevents redundant queries for papers assigned to multiple team members, thereby reducing unnecessary requests.

##### 1.1.3. Batch Querying
Instead of sending a separate request for each paper, the program uses batch querying, which combines multiple arXiv IDs into a single API request.

For example, with a batch size of 50, each request can retrieve metadata for up to 50 papers through the id_list parameter.
With 20,000 IDs and a batch size of 50, the theoretical number of metadata requests is:

With 20,000 IDs and a batch size of 50, the theoretical number of metadata requests is:

$$
N_{\text{metadata requests}}
=\left\lceil\frac{20000}{50}\right\rceil
=400
$$

Compared with a batch size of 20, which requires 1,000 metadata requests, increasing the batch size to 50 reduces the expected number of requests to 400, representing a 60% reduction.

The program uses the following configuration:

- Metadata batch size: 50.

- Revision batch size: 50.

- Delay: 3.5 seconds.

- Cooldown: 600 seconds.

- Number of concurrent HTTP connections: 1.

Requests are executed sequentially. The program starts the next request only after the previous request has completed and the configured delay requirement has been satisfied.


##### 1.1.4. Metadata Scarping Using the arXiv API
The metadata collection process consists of the following steps:

1. Read the list of IDs from group_ids.txt using load_ids_file().

2. Divide the IDs into batches containing at most 50 IDs.

3. Check whether a metadata checkpoint already exists for the current batch.

4. If the data are not available locally, send an HTTP request using ArxivClient.fetch().

5. Receive the API response in Atom XML format.

6. Extract metadata fields using parse_metadata().

7. Save the batch metadata to a checkpoint file.

|Field|Description|
|:--|:--|
|arxiv_id|Unique identifier of the paper|
|title|Paper title|
|authors|List of paper's authors|
|submission_date|Initial submission date|
|latest_updated|Update date of the latest version|
|revised_dates|List of revision dates|
|venue|Conference or journal information, if available|
|latest_version|Number of the latest version|
|versions|List of version identifiers|
|abstract|Paper abstract|
|status|Metadata collection status|

```json
{
  "arxiv_id": "[paper_id]",
  "title": "[paper_title]",
  "authors": [
    "[author_1], [author_2],..."
  ],
  "submission_date": "[initial_date]",
  "latest_updated": "[latest_version_update]",
  "revised_dates": [
    "[revised_date_of_version]"
  ],
  "venue": null,
  "latest_version": [number_of_latest_version],
  "versions": [
    "[paper_idv1]",
    "[paper_idv2]", 
    ...
    "[paper_idvN]"
  ],
  "abstract": "[paper_abstract]",
  "status": "[found]"
}

```



The module is implemented by Python, using `urllib.request` to send HTTP requests and `xml.etree.ElementTree` to parse Atom XML responses.

The `json`, `pathlib`, `argparse`, `hashlib`, `time`, and `datetime` libraries support data storage, validation, configuration management, and performance measurement.


##### 1.1.5. Entry Discovery Performance Optimization
During development, the team has been through many optimization strategies:
|Version|Description|
|:--|:--|
|1|Retrieves metadata and revision dates for all versions from v2 to vN|
|2|Reuses latest_updated to avoid querying the latest version again|
|3|Increases the batch size to 50 to reduce API requests|
|4|Collects all metadata first, then groups missing revision IDs across the entire dataset|

In the previous approach, the crawler retrieved revision dates immediately after processing each metadata batch. This could produce many small requests because each metadata batch might contain only a few versions requiring additional queries.

In version 4, the team reorganize the workflow into two main phases:
- **Phase 1:** The crawler retrieves metadata for the entire ID list and saves checkpoints after processing each batch.
- **Phase 2:** The crawler reuses latest_updated, identifies versions with missing revision dates, and groups them by version number (v2, v3, v4, etc.) across the entire dataset.

Each revision group is then divided into requests containing at most 50 IDs.

This strategy reduces request fragmentation and improves the utilization of the arXiv API's batch querying capability.

During the version 4 run involving 20,000 IDs, the following statistics were recorded:

|Metric|Value|
|:--|:--|
|Total IDs processed|20,000|
|Number of metadata batches|400|
|Latest-version dates reused|5,716|
|Revision IDs still requiring queries when the queue was created|1,930|
|Number of version layers|15|




#### 1.2. Version Detection Mechanism

##### 1.2.1. Purpose of Version Detection

A paper on arXiv may be updated multiple times by its authors. Each update creates a new version, represented by v1, v2, and v3.

The Entry Discovery module must identify all published versions so that the Source Download module can retrieve the complete source files for every version.

##### 1.2.2. Version Detection Algorithm

When an arXiv ID is queried without a version suffix, the API returns metadata associated with the latest version.

For example: 2511.11735 -> 2511.11735v3

From the v3 suffix, the crawler determines that the paper has three versions.

In `discovery.py`, the `parse_metadata()` function uses a Regex to extract the version information:

```python
VERSIONED = re.compile(r'(\d{4}\.\d{4,5})v(\d+)$')
```

In [ ]:
VERSIONED = re.compile(r'(\d{4}\.\d{4,5})v(\d+)$')

This expression separates the identifier into two components:

- The first component (\d{4}\.\d{4,5}) represents the base arXiv ID.

- The second component (\d+) represents the version number.

For example, given 2511.11735v3, the extracted values are:

- arxiv_id = "2511.11735"

- latest_version = 3

The crawler then generates the complete version list:
```python
["2511.11735v1", "2511.11735v2", "2511.11735v3"]
```

The algorithm relies on arXiv's sequential version numbering convention, from v1 to the latest available version.

##### 1.2.3. Identifying Submission Dates and Revision Dates

In addition to detecting the number of versions, the crawler must determine the submission or update date associated with each version.

For a paper with (n) versions:

- submission_date: The original submission date, corresponding to v1.

- revised_dates: The list of update dates for versions from v2 to vN.

- latest_updated: The update date of the latest version.

The required number of revision dates is:
$
[
N_{\text{revised dates}}=n-1
]
$

In version 4, the crawler uses latest_updated to obtain the date of vN without sending an additional request for the latest version.

For intermediate versions whose dates are not available in the cache, the crawler sends requests containing specific version IDs and extracts the updated field from the XML responses.

The retrieved dates are stored in `revised_dates_cache.json` and subsequently merged with the metadata using `attach_revised_dates()`.



## 1.3. Output Files and Data Structure

### 1.3.1. Overview of the Output Structure

The Entry Discovery module produces structured metadata and version information for the assigned arXiv papers. The collected data are first stored in the `data/discovery/` directory. After the crawling process is completed, the `prepare_papers_handoff.py` script converts the results into a paper-based directory structure for subsequent modules.

The output is organized into two main directories:

- **`data/discovery/`**: Stores the crawling results, metadata checkpoints, revision-date cache, execution status, error logs, and benchmark statistics.
- **`data/output/`**: Stores individual paper metadata files, version directories, and an index mapping arXiv IDs to their available versions.

The overall data flow is:

```text
data/input/group_ids.txt
          |
          v
   smart_discovery.py
          |
          v
data/discovery/
    discovered_papers_with_revised_dates.json
          |
          v
 prepare_papers_handoff.py
          |
          v
data/output/
    papers_index.json
    papers/
```

This separation ensures that intermediate crawling data are preserved while the final output is organized into a format that can be used directly by the Source Download and Reference Crawling modules.

### 1.3.2. Main Discovery Output: `discovered_papers_with_revised_dates.json`

The primary output file of the Entry Discovery crawler is:

`data/discovery/discovered_papers_with_revised_dates.json`

This file is generated by `smart_discovery.py` after combining the collected metadata with revision dates stored in `revised_dates_cache.json`.

The file contains a JSON array of paper objects. Each object represents one successfully discovered arXiv paper.

The actual metadata schema is defined in `parse_metadata()` and completed using `attach_revised_dates()`.

| Field | Data Type |
|---|---|---|
| `arxiv_id` | String |
| `title` | String |
| `authors` | List[String] |
| `submission_date` | String |
| `latest_updated` | String |
| `revised_dates` | List[String] or null |
| `venue` | String or null |
| `latest_version` | Integer | 
| `versions` | List[String] | 
| `abstract` | String | 
| `status` | String | 

An example paper record is shown below:

```json
{
  "arxiv_id": "2511.11721",
  "title": "A Meta-Heuristic Load Balancer for Cloud Computing Systems",
  "authors": [
    "Leszek Sliwko",
    "Vladimir Getov"
  ],
  "submission_date": "2025-11-13T00:26:21Z",
  "latest_updated": "2025-11-18T02:24:32Z",
  "revised_dates": [
    "2025-11-18T02:24:32Z"
  ],
  "venue": "In 2015 IEEE 39th Annual Computer Software and Applications Conference, vol. 3, pp. 121-126. IEEE, 2015",
  "latest_version": 2,
  "versions": [
    "2511.11721v1",
    "2511.11721v2"
  ],
  "abstract": "This paper presents a strategy ...",
  "status": "found"
}

```


The `versions` field is generated based on `latest_version`. For instance, if `latest_version = 3`, the crawler generates the identifiers `v1`, `v2`, and `v3`.

The `revised_dates` field is constructed by `attach_revised_dates()` using the revision-date cache. Its behavior depends on the paper's version history:

- For a paper with only `v1`, `revised_dates` is an empty list (`[]`).
- For a paper with multiple versions and all revision dates available, it contains the ordered dates from `v2` to `vN`.
- If one or more required revision dates are missing, `revised_dates` is set to `null`.

This approach prevents incomplete revision histories from being incorrectly treated as complete.

The output file contains metadata only. It does not contain downloaded LaTeX source files, BibTeX files, or reference metadata.

### 1.3.3. Checkpoint, Cache, and Monitoring Files

In addition to the main JSON output, the crawler generates several supporting files to enable recovery, validation, and performance monitoring.

The directory structure is:

```text
data/
└── discovery/
    ├── batches/
    │   ├── batch_00000.json
    │   ├── batch_00001.json
    │   └── ...
    │
    ├── partial_batches/
    ├── checkpoint_manifest.json
    ├── revised_dates_cache.json
    ├── discovered_papers_with_revised_dates.json
    ├── crawl_queue.json
    ├── crawl_attempts.json
    ├── crawl_errors.json
    ├── crawl_errors.jsonl
    ├── smart_summary.json
    └── benchmark_sessions.jsonl
```


Each file serves a specific purpose:

| File | Purpose |
|---|---|
| `batches/batch_XXXXX.json` | Stores completed metadata batches |
| `partial_batches/` | Stores partially completed metadata batches |
| `checkpoint_manifest.json` | Records the ID-list hash, total number of IDs, and batch size |
| `revised_dates_cache.json` | Maps version IDs to their retrieved update dates |
| `crawl_queue.json` | Tracks completion status and missing versions for each paper |
| `crawl_attempts.json` | Records failed-task attempt counts |
| `crawl_errors.json` | Stores the latest recorded errors by task |
| `crawl_errors.jsonl` | Maintains a chronological error log |
| `smart_summary.json` | Stores overall crawling completion statistics |
| `benchmark_sessions.jsonl` | Records performance metrics for each execution session |

**Metadata checkpoints**

Metadata are saved after each completed batch rather than waiting until all 20,000 IDs have been processed.

With a batch size of 50, a full set of 20,000 IDs is divided into 400 metadata batches. The corresponding files are named `batch_00000.json` through `batch_00399.json`.

When the crawler is restarted, existing valid checkpoints can be reused, avoiding unnecessary metadata requests.

**Revision-date cache**

The `revised_dates_cache.json` file stores revision dates using version IDs as keys:

```json
{
  "2609.12834v2": "2026-09-10T10:00:00Z",
  "2609.12834v3": "2026-09-20T10:00:00Z"
}
```

Before requesting revision data, crawler checks this cache and includes only missing version IDs in the global revision queue.

**Crawling summary and benchmark**

The `smart_summary.json` file provides the overall completion status, including the number of IDs, discovered papers, completed tasks, and pending papers.

The `benchmark_sessions.jsonl` file records metrics such as execution time, API request count, HTTP 429 responses, network time, pacing time, and cooldown time.

These files support both error recovery and performance evaluation without requiring the crawler to repeat completed work.


#### 1.4. Edge Cases & Exception Handling in Module 1

##### 1.4.1. Timeouts, Network Errors, and HTTP Errors
ArxivClient uses a default timeout of 40 seconds. As instantiated by `smart_discovery.py`, it allows up to two retries after the first attempt for network failures and retryable server errors, with increasing wait intervals. HTTP 4xx responses other than 429 are treated as non-retryable. The client records request, retry, and network-time metrics.

The Global crawler records handled request failures in crawl_errors.json, crawl_attempts.json, and crawl_errors.jsonl, while keeping unresolved work pending. The code does not uniformly retry every XML parsing exception; therefore, the report should not claim that all malformed XML responses are automatically recovered.


##### 1.4.4. HTTP 429 and Cooldown
The HTTP client raises RateLimitError on status 429. The Global crawler then schedules a cooldown at least as long as --cooldown, or longer if a valid numeric Retry-After value is available. The deadline is written to rate_limit_until_utc.json.

Unless --no-wait-cooldown is set or the consecutive-429 threshold is reached, the crawler waits and retries the same task after cooldown. In the recorded run, one 429 event caused approximately 600 seconds of cooldown. The single failed_tasks event did not leave pending work at completion.